# AT1 — Perceptron e KNN em prática

**Aluno:** Bruno  
**Disciplina:** Aprendizagem de Máquina — DSM

Implementação dos três desafios com Python e NumPy, sem modelos prontos de bibliotecas de aprendizado de máquina.

[Enunciado do professor](https://i-davies.github.io/conteudo-fatec-2026-2/AM/05_AT1_Perceptron_KNN/01_Atividade_Avaliativa/)

Selecione o Python da pasta `.venv` e execute **Restart Kernel and Run All Cells**. Os dados estão no notebook; não há arquivos externos a carregar.

In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
print(f"NumPy: {np.__version__}")

NumPy: 2.5.3


## 1. Perceptron treinável — triagem de fraude

Cada linha contém **valor normalizado da transação** e **frequência de operações**. Os rótulos são 0 (legítima) e 1 (suspeita).

O modelo calcula $z = x \cdot w + b$. Adotamos o degrau $\hat y = 1$ quando $z \geq 0$, e 0 caso contrário.

A cada erro, a regra de Rosenblatt atualiza $w \leftarrow w + \eta(y-\hat y)x$ e $b \leftarrow b + \eta(y-\hat y)$. Uma época corresponde a uma passagem pela base. Usamos taxa 0,1 e limite de 100 épocas, com parada antecipada quando uma época termina sem erros. A ordem das amostras permanece fixa para reprodução.

In [2]:
X_treino_fraude = np.array([
    [1.5, 1.0], [2.0, 2.0], [3.5, 1.5], [3.0, 3.0],
    [6.5, 5.0], [7.0, 7.0], [8.5, 6.0], [9.0, 8.0]
])
y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])
transacoes = np.array([[2.5, 2.0], [8.0, 6.5]])

In [3]:
def predizer_perceptron(amostra, pesos, vies):
    """Aplica a combinação linear e o degrau a uma amostra."""
    return int(np.dot(amostra, pesos) + vies >= 0)


def treinar_perceptron(X, y, taxa_aprendizado, epocas):
    """Treina sequencialmente pela regra de Rosenblatt e retorna w e b."""
    pesos = np.zeros(X.shape[1], dtype=float)
    vies = 0.0
    for _ in range(epocas):
        erros = 0
        for amostra, classe_real in zip(X, y):
            previsao = predizer_perceptron(amostra, pesos, vies)
            erro = classe_real - previsao
            if erro != 0:
                # Todos os pesos são atualizados juntos pelo vetor da amostra.
                pesos += taxa_aprendizado * erro * amostra
                vies += taxa_aprendizado * erro
                erros += 1
        if erros == 0:
            break
    return pesos, vies

### Exemplo de uma atualização

Na primeira amostra, `[1.5, 1.0]`, os pesos e o bias começam em zero. Assim, $z=0$ e o degrau prevê 1. Como o rótulo é 0, o erro é −1. Com taxa 0,1, os pesos passam a `[-0.15, -0.10]` e o bias a `-0.10`. As próximas amostras usam esses parâmetros já atualizados.

O laço de treino é intencional: cada atualização depende da anterior. O produto escalar e a atualização do vetor de pesos usam NumPy.

In [4]:
pesos, vies = treinar_perceptron(X_treino_fraude, y_treino_fraude, 0.1, 100)
print("Pesos finais:", pesos)
print(f"Bias final: {vies:.4f}")

rotulos_fraude = {0: "Transação Legítima", 1: "Transação Suspeita (Risco de Fraude)"}
previsoes_fraude = []
for nome, amostra in zip(["A", "B"], transacoes):
    classe = predizer_perceptron(amostra, pesos, vies)
    previsoes_fraude.append(classe)
    score = np.dot(amostra, pesos) + vies
    print(f"Transação {nome}: {rotulos_fraude[classe]} | classe={classe} | z={score:.4f}")

Pesos finais: [0.05 0.1 ]
Bias final: -0.8000
Transação A: Transação Legítima | classe=0 | z=-0.4750
Transação B: Transação Suspeita (Risco de Fraude) | classe=1 | z=0.2500


### Interpretação

A transação A tem valores menores nas duas características, enquanto B se aproxima do grupo suspeito. A saída é uma classe, e o valor de $z$ **não é uma probabilidade de fraude**. A validação na base de treinamento ao final apenas confirma que a implementação separa os oito exemplos; não mede desempenho em transações reais.

## 2. KNN — risco de cancelamento

As características são **dias de inatividade** e **chamados críticos em aberto**. Com $K=3$, a previsão é a classe mais frequente entre os três clientes históricos mais próximos.

Distância Euclidiana: $d(x,q)=\sqrt{\sum_j(x_j-q_j)^2}$.  
Distância Manhattan: $d(x,q)=\sum_j|x_j-q_j|$.

O broadcasting subtrai a consulta de todas as linhas; `axis=1` soma as características de cada linha. Os índices exibidos começam em **0**. Em distâncias empatadas, preservamos a ordem da base; em empate de votos, a menor classe vence. Para este conjunto binário e $K=3$, não há empate de votos.

In [5]:
X_treino_churn = np.array([
    [2.0, 1.0], [3.0, 0.0], [5.0, 1.0], [6.0, 2.0],
    [15.0, 4.0], [18.0, 5.0], [20.0, 4.0], [22.0, 6.0]
])
y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])
clientes = np.array([[4.0, 1.0], [17.0, 5.0]])

In [6]:
def calcular_distancias(X, consulta, metrica="euclidiana"):
    """Calcula a distância a todas as linhas, sem laços manuais."""
    diferencas = np.asarray(X, dtype=float) - np.asarray(consulta, dtype=float)
    if metrica == "euclidiana":
        return np.sqrt(np.sum(diferencas ** 2, axis=1))
    if metrica == "manhattan":
        return np.sum(np.abs(diferencas), axis=1)
    raise ValueError("Métrica deve ser 'euclidiana' ou 'manhattan'.")


def classificar_knn(X, y, consulta, k=3, metrica="euclidiana"):
    """Retorna classe majoritária, índices e distâncias dos k vizinhos."""
    if not isinstance(k, (int, np.integer)) or not 1 <= k <= len(X):
        raise ValueError("k deve ser inteiro entre 1 e o tamanho da base.")
    distancias = calcular_distancias(X, consulta, metrica)
    indices = np.argsort(distancias, kind="stable")[:k]
    classes, votos = np.unique(np.asarray(y)[indices], return_counts=True)
    classe = int(classes[np.argmax(votos)])
    return classe, indices, distancias[indices]

In [7]:
rotulos_churn = {0: "Baixo Risco", 1: "Alto Risco"}
for metrica in ["euclidiana", "manhattan"]:
    print(f"\nMétrica: {metrica}")
    for numero, consulta in enumerate(clientes, start=1):
        classe, indices, distancias = classificar_knn(
            X_treino_churn, y_treino_churn, consulta, k=3, metrica=metrica
        )
        print(f"Cliente {numero}: {rotulos_churn[classe]} (classe {classe})")
        print(f"  Índices (base 0): {indices}")
        print(f"  Distâncias: {distancias}")
        print(f"  Votos dos vizinhos: {y_treino_churn[indices]}")


Métrica: euclidiana
Cliente 1: Baixo Risco (classe 0)
  Índices (base 0): [2 1 0]
  Distâncias: [1.     1.4142 2.    ]
  Votos dos vizinhos: [0 0 0]
Cliente 2: Alto Risco (classe 1)
  Índices (base 0): [5 4 6]
  Distâncias: [1.     2.2361 3.1623]
  Votos dos vizinhos: [1 1 1]

Métrica: manhattan
Cliente 1: Baixo Risco (classe 0)
  Índices (base 0): [2 0 1]
  Distâncias: [1. 2. 2.]
  Votos dos vizinhos: [0 0 0]
Cliente 2: Alto Risco (classe 1)
  Índices (base 0): [5 4 6]
  Distâncias: [1. 3. 4.]
  Votos dos vizinhos: [1 1 1]


### Interpretação

O Cliente 1 fica próximo dos exemplos de baixo risco e o Cliente 2 dos exemplos de alto risco. As duas métricas podem produzir distâncias e ordenações diferentes, mesmo quando a classe majoritária é igual. O KNN armazena os exemplos e consulta a vizinhança; não ajusta pesos como o Perceptron.

## 3. Recomendação de servidores por similaridade

Cada servidor é um ponto com **vCPUs, RAM (GB) e SSD (GB)**. A consulta é `[12, 28, 450]`, e retornamos os dois menores valores de distância Euclidiana. Aqui não existe vetor de classes nem votação.

In [8]:
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],
    [4.0, 8.0, 100.0],
    [8.0, 16.0, 250.0],
    [16.0, 32.0, 500.0],
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0]
])
nomes_servidores = [
    "Micro Instância Web", "Standard App Server", "Medium Backend & Cache",
    "Database Enterprise", "High Performance Computing", "Big Data & AI Training"
]
perfil_demandado = np.array([12.0, 28.0, 450.0])


def recomendar_servidores(catalogo, perfil, k=2, metrica="euclidiana"):
    """Retorna índices e distâncias dos servidores mais similares."""
    if metrica != "euclidiana":
        raise ValueError("O recomendador utiliza a métrica euclidiana.")
    if not isinstance(k, (int, np.integer)) or not 1 <= k <= len(catalogo):
        raise ValueError("k deve ser inteiro entre 1 e o tamanho do catálogo.")
    distancias = calcular_distancias(catalogo, perfil, metrica)
    indices = np.argsort(distancias, kind="stable")[:k]
    return indices, distancias[indices]

In [9]:
indices_servidores, distancias_servidores = recomendar_servidores(
    catalogo_servidores, perfil_demandado, k=2, metrica="euclidiana"
)
print("Demanda: 12 vCPUs | 28 GB RAM | 450 GB SSD\n")
for posicao, (indice, distancia) in enumerate(
    zip(indices_servidores, distancias_servidores), start=1
):
    cpu, ram, ssd = catalogo_servidores[indice]
    print(f"{posicao}º lugar — {nomes_servidores[indice]}")
    print(f"  {cpu:g} vCPUs | {ram:g} GB RAM | {ssd:g} GB SSD")
    print(f"  Distância Euclidiana: {distancia:.4f}\n")

Demanda: 12 vCPUs | 28 GB RAM | 450 GB SSD

1º lugar — Database Enterprise
  16 vCPUs | 32 GB RAM | 500 GB SSD
  Distância Euclidiana: 50.3190

2º lugar — Medium Backend & Cache
  8 vCPUs | 16 GB RAM | 250 GB SSD
  Distância Euclidiana: 200.3996



### Interpretação e limites

Mantivemos as escalas originais do enunciado. Como o SSD tem valores numericamente maiores, tende a influenciar mais a distância. Normalização seria uma escolha relevante em outra análise, mas alteraria a comparação solicitada.

Similaridade não garante capacidade mínima: o segundo servidor recomendado pode ficar abaixo da demanda. Para exigir atendimento integral, seria necessário filtrar recursos mínimos antes de ordenar. O desafio pede apenas proximidade geométrica.

## 4. Verificação da execução

Estas verificações conferem os exemplos, a fórmula das distâncias e a ordenação. Execute tudo com um kernel reiniciado para garantir que nenhuma célula depende de variáveis de uma execução anterior.

In [10]:
# Distâncias conhecidas: triângulo 3-4-5 e soma absoluta 3+4.
np.testing.assert_allclose(calcular_distancias([[3, 4]], [0, 0]), [5])
np.testing.assert_allclose(calcular_distancias([[3, 4]], [0, 0], "manhattan"), [7])

predicoes_treino = (X_treino_fraude @ pesos + vies >= 0).astype(int)
np.testing.assert_array_equal(predicoes_treino, y_treino_fraude)
np.testing.assert_array_equal(previsoes_fraude, [0, 1])

for metrica in ["euclidiana", "manhattan"]:
    for consulta, esperado in zip(clientes, [0, 1]):
        classe, indices, distancias = classificar_knn(
            X_treino_churn, y_treino_churn, consulta, 3, metrica
        )
        assert classe == esperado
        assert len(indices) == 3
        assert np.all(np.diff(distancias) >= 0)

np.testing.assert_array_equal(indices_servidores, [3, 2])
np.testing.assert_allclose(distancias_servidores, np.sqrt([2532, 40160]))
print("Validações concluídas: treino, previsões, distâncias e ranking corretos.")

Validações concluídas: treino, previsões, distâncias e ranking corretos.
